# Baseline Comparison on `ap_burst2k`

Replaces `05_baselines_comparison.ipynb`, which runs on `data/raw/llm_generated` -- the
benchmark a per-record content probe solves at AUROC 1.0000. Every baseline number in that
notebook describes a withdrawn benchmark, and the comparison there was uninformative in
principle: when every model scores 1.0000, a comparison has no content.

**Six baselines, all measured on the same 500 test sequences as the detector.**

Everything is at **sequence level** -- one observation per sequence, which is the granularity
the question "is this agent's memory poisoned?" is asked at. Window-level numbers are not
comparable: a position-only predictor scores 0.6086 at window level on this benchmark,
because `n_nodes(snapshot i) = (i + 1) * snapshot_interval` makes node count an exact linear
function of window position. At sequence level every sequence has identical length and node
count, and `n_nodes` scores exactly 0.5000.

This notebook loads artefacts already on disk and opens in seconds. Each section states the
command that produced its artefact.

## 1. Setup

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import yaml
from sklearn.metrics import roc_auc_score, average_precision_score

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
import os
os.chdir(REPO_ROOT)

from src.experiments.sequence_eval import boot_ci, paired_boot

DATA_DIR = REPO_ROOT / "data" / "raw" / "ap_burst2k"
EXP = REPO_ROOT / "experiments"
BASE = EXP / "run_burst2k_base"
CONFIG = yaml.safe_load(open(REPO_ROOT / "configs" / "default.yaml"))

# Every model is scored on the SAME 500 test sequences, in the same order. Each run's
# sequence_scores_test.npz carries its own sequence_id and y, and they are asserted equal
# before anything is paired -- comparing two models scored on different orderings produces a
# number that looks fine and means nothing.
base = np.load(BASE / "sequence_scores_test.npz", allow_pickle=True)
Y = base["y"]
ORDER = base["sequence_id"]
DETECTOR = base["T-GNN_baseline_(window_max)"]
N_BOOT = 2000

print(f"{len(Y)} test sequences, {int(Y.sum())} poisoned")
print(f"detector (T-GNN, window max): AUROC {roc_auc_score(Y, DETECTOR):.4f}  "
      f"AP {average_precision_score(Y, DETECTOR):.4f}")


def load_scores(run_dir, key_filter=None, filename="sequence_scores_test.npz"):
    """Per-sequence scores from a run, with the ordering checked against the detector's.

    key_filter matters more than it looks. Every run directory written by sequence_eval also
    contains that run's copy of the LOGISTIC REGRESSION baselines, because sequence_eval
    computes them every time. Loading a static-GNN run without filtering therefore pulls in
    three regression rows and labels them as static-GNN results.
    """
    p = Path(run_dir) / filename
    if not p.exists():
        return None
    z = np.load(p, allow_pickle=True)
    assert (z["sequence_id"] == ORDER).all(), f"{p} is ordered differently -- cannot pair"
    assert (z["y"] == Y).all(), f"{p} has different labels -- different split?"
    return {k: z[k] for k in z.files
            if k not in ("sequence_id", "y") and (key_filter is None or key_filter in k)}


def row(name, scores, expected=None, pair=True):
    auroc, apr = roc_auc_score(Y, scores), average_precision_score(Y, scores)
    lo, hi = boot_ci(Y, scores, N_BOOT)
    out = f"{name:<40} {auroc:>7.4f} {apr:>7.4f}  [{lo:.4f}, {hi:.4f}]"
    if pair:
        d, dlo, dhi = paired_boot(Y, DETECTOR, scores, N_BOOT, metric="auroc")
        sig = "*" if (dlo > 0 or dhi < 0) else " "
        out += f"  {d:+.4f}{sig}"
    if expected:
        out += f"   exp {expected[0]:.4f}/{expected[1]:.4f}"
    print(out)
    return auroc, apr


HEAD = (f"{'model':<40} {'AUROC':>7} {'AP':>7}  {'95% CI':>18}  "
        f"{'dAUROC':>8}\n" + "-" * 104)

## 2. Baselines 1-3: logistic regression on hand-built features

Three feature sets, fitted on the training sequences and scored on test.

**`all features (13)` is an ORACLE.** Its features were hand-built by inspecting how the
attack is generated -- burst sharpness, growth-rate jumps, how many snapshots share the final
snapshot's max degree. Beating it is a strong claim. *Tying* it is not a failure, and the
thesis should say so rather than apologise for it.

`final-state features (6)` is the control that matters for benchmark validity: it sits at
chance by construction, because the matched benign cluster gives clean sequences the same
final density a flood produces.

In [ ]:
print("produced by: python -m src.experiments.sequence_eval --data data/raw/ap_burst2k \\")
print("               --run-dir experiments/run_burst2k_base --label 'T-GNN baseline'\n")
print(HEAD)
EXPECT = {"final-state features (6)": (0.4650, 0.3872),
          "temporal features (7)": (0.7934, 0.7108),
          "all features (13)": (0.8030, 0.7203)}
lr = load_scores(BASE)
for key, exp in EXPECT.items():
    row(key, lr[key.replace(" ", "_")], exp)
print("\n* = 95% CI of the paired difference against the detector excludes zero")

## 3. Baselines 4-5: Isolation Forest and One-Class SVM

Unsupervised. Neither sees a label at fit time; labels are used only to score the output.

Reported at **two levels**, because they answer different questions and the answers differ.

**Record level** -- "are injected records distributional outliers?" This is the claim from
Sections 4.6 and 6.3 that survives the amendment, and it rests on firmer ground here than it
ever did on the old benchmark. On `ap_burst2k` every record, benign or poisoned, is a genuine
unmodified StrategyQA knowledge-base entry, so there is no text-level property to be an
outlier in.

**Sequence level** -- fitted on the benign *training* sequences only, which is the realistic
deployment: an operator has clean history and no labelled attacks.

Both beat chance at sequence level. State why, or it reads as inconsistent with the record
level: they are fitted on the 13-feature table, so they are unsupervised *detectors* handed
engineered *features* that already encode what the attack looks like.

In [ ]:
print("produced by: python -m src.experiments.unsupervised_baselines --data data/raw/ap_burst2k\n")
unsup_path = DATA_DIR / "unsupervised_baselines.json"
if unsup_path.exists():
    unsup = json.load(open(unsup_path))
    print("RECORD LEVEL -- base rate AP 0.0250, so AP is the informative column")
    if not unsup["record_level"]:
        print("  PENDING -- rerun WITHOUT --skip-record")
    for name, r in unsup["record_level"].items():
        print(f"  {name:<22} AUROC {r['auroc']:.4f}   AP {r['ap']:.4f}")
    print("  expected: Isolation Forest 0.5029 / 0.0275, One-Class SVM 0.5531 / 0.0284")
    print("  Both at or near chance. The One-Class SVM's 0.5531 is marginally above it --")
    print("  write 'at or near chance', not 'at chance'.\n")
else:
    print(f"  PENDING -- {unsup_path} not found")

print("SEQUENCE LEVEL")
print(HEAD)
u = load_scores(DATA_DIR, filename="unsupervised_sequence_scores_test.npz") or {}
EXPECT_U = {"Isolation_Forest": (0.6538, 0.5449), "One-Class_SVM": (0.5719, 0.5044)}
for key, exp in EXPECT_U.items():
    if key in u:
        row(key.replace("_", " "), u[key], exp)
    else:
        print(f"{key.replace('_', ' '):<40} PENDING")

## 4. Baseline 6: static GNN -- the temporal components removed

**The most informative baseline here.** `ap_burst2k` is built so a single snapshot carries no
signal, so a model reading only the last snapshot should sit near chance while the full
detector reaches 0.81. That isolates the temporal components using the thesis's own published
baseline rather than a feature set chosen after the fact.

It trains through the **same `Trainer`** as the detector: same loss, optimiser, batch size,
augmentation, epoch budget and validation-AUROC checkpoint selection, same input features,
same attention readout. The only difference is that `NodeMemoryModule` and
`TemporalAggregator` are gone.

Run under **both** readouts. With mean pooling alone, a loss could be blamed on the readout
rather than on the missing temporal component -- and Sections 2.6 and 3.8 argue that mean
pooling attenuates exactly this kind of small anomalous subgraph. The result holds either way,
which is what closes that objection.

**Concede this in the write-up:** the static model has 306,243 parameters against the
detector's 670,915. You cannot remove the temporal machinery and hold parameter count fixed,
so "it is smaller" is an available counter-explanation. Say it before an examiner does.

In [ ]:
print("produced by: python -m src.experiments.train_static_baseline --data data/raw/ap_burst2k \\")
print("               --output experiments/run_burst2k_static_attn --readout attention \\")
print("               --epochs-supervised 12 --seed 42\n")
print(HEAD)
EXPECT_S = {("attn", "attention"): (0.5602, 0.4684), ("mean", "mean"): (0.5424, 0.4652)}
for (tag, readout), exp in EXPECT_S.items():
    s = load_scores(EXP / f"run_burst2k_static_{tag}")
    if s is None:
        print(f"{'static GNN (' + readout + ')':<40} PENDING")
        continue
    for agg in ("max", "mean", "last"):
        key = f"static_GNN_({readout})_(window_{agg})"
        if key in s:
            row(key.replace("_", " "), s[key], exp if agg == "max" else None)

## 5. The comparison table

Sorted by AUROC. This is the table for Chapter 4.

In [ ]:
# Each source is filtered to the rows it actually owns. Without the filters the static runs
# contribute their bundled copies of the regression baselines a second and third time, under
# the wrong label -- a duplicated row with the wrong kind beside it is worse than a gap.
rows = []
for key, s in (load_scores(BASE) or {}).items():
    kind = "proposed" if key.startswith("T-GNN") else "logistic regression"
    rows.append((key.replace("_", " "), kind, s))
for key, s in (load_scores(DATA_DIR,
                           filename="unsupervised_sequence_scores_test.npz") or {}).items():
    rows.append((key.replace("_", " "), "unsupervised", s))
for tag in ("attn", "mean"):
    for key, s in (load_scores(EXP / f"run_burst2k_static_{tag}",
                               key_filter="static_GNN") or {}).items():
        rows.append((key.replace("_", " "), "static GNN (no temporal)", s))

seen = set()
deduped = []
for name, kind, s in rows:
    if name in seen:
        continue
    seen.add(name)
    deduped.append((name, kind, s))
rows = deduped

scored = sorted(((n, k, roc_auc_score(Y, s), average_precision_score(Y, s))
                 for n, k, s in rows), key=lambda r: -r[2])
print(f"{'model':<40} {'kind':<26} {'AUROC':>7} {'AP':>7}")
print("-" * 84)
for n, k, a, p in scored:
    print(f"{n:<40} {k:<26} {a:>7.4f} {p:>7.4f}")

print("""
HOW TO READ THIS

The only baselines that match the detector on AUROC are the temporal and 13-feature logistic
regressions, and both are ORACLES built from knowledge of how the attack is generated. Every
baseline not given that knowledge is beaten significantly on BOTH metrics:

  static GNN        +0.2517 AUROC  [+0.2049, +0.2995]   +0.3464 AP  [+0.2927, +0.4022]
  Isolation Forest  +0.1592 AUROC  [+0.1009, +0.2171]   +0.2715 AP  [+0.2070, +0.3378]
  One-Class SVM     +0.2407 AUROC  [+0.1801, +0.3012]   +0.3097 AP  [+0.2459, +0.3765]

That is the claim to make: the detector matches a baseline that was told the answer, and beats
everything that was not. It is a better claim than a bare number, and it answers the AUROC tie
against the oracle in the same breath.

Nothing here is at 1.0000, and nothing should be. The withdrawn benchmark produced a perfect
score for every model, which is precisely how it was found to be defective. A spread across
models is evidence; a ceiling is the absence of it.
""")